# 0. Imports

In [1]:
import pandas as pd
import numpy as np
import gurobipy as gp

# 1. Data exploration

The initial MILP formulation of the problem is given by:

$$\min\sum_{k\in K}\sum_{i\in N}\sum_{j\in N}b_1 c^kt_{ij}x_{ij}^k+\sum_{i\in P}b_2c^0(1-\sum_{k\in K}\sum_{j\in P\cup D}x_{ij}^k)+b_3E[Q(x,w,a,q,\xi)]$$

In this formulation, $x_{ij}^k$ represents the decision variable (indicator whether arc $ij$ is travelled by vehicle $k$). To start the problem, the travel times between nodes will have to be extracted and associated with a travel time.

## 1.1 Travel times 15-17:

In [2]:
travel_times_15_17 = pd.read_csv("travel_times_15_17.csv", index_col=0)

In [3]:
travel_times_15_17.head(15)

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589152,"Fribourg, Mon-Repos",46.806711,7.172136,270,35,0.58,42,0.70,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589138,"Fribourg, Cité-Jardins",46.809385,7.170446,659,86,1.43,117,1.95,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.30,174,2.90,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8587255,"Fribourg, Tilleul/Cathédrale",46.806090,7.161261,3788,445,7.42,506,8.43,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589161,"Fribourg, St-Pierre",46.803911,7.155266,4335,561,9.35,622,10.37,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8592374,"Fribourg/Freiburg, Pl. Gare",46.802898,7.151410,4719,695,11.58,749,12.48,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589130,"Villars-sur-Glâne, Méridienne",46.794173,7.111828,9363,808,13.47,866,14.43,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589131,"Villars-sur-Glâne, Moncor",46.798570,7.120788,8377,699,11.65,754,12.57,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8588344,"Villars-sur-Glâne, Belle-Croix",46.800233,7.125455,8261,722,12.03,757,12.62,OK


In [4]:
origins_15_17 = pd.unique(travel_times_15_17["origin_name"])
origins_15_17

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [5]:
print(travel_times_15_17['origin_name'].value_counts())
print(travel_times_15_17['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  196
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196


In [6]:
print(travel_times_15_17["dest_name"].value_counts())
print(travel_times_15_17["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Fribourg, Chaley                  196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196
Fr

In [7]:
print(travel_times_15_17.value_counts(['origin_name', 'dest_name']))
print(travel_times_15_17.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             7
                       Fribourg, Cité-Jardins          7
                       Fribourg, Boschung              7
                       Fribourg, Tilleul/Cathédrale    7
                       Fribourg, St-Pierre             7
                                                      ..
Givisiez, Mont Carmel  Fribourg, Industrie             7
                       Fribourg, J. Vogt               7
                       Fribourg, Fries                 7
                       Fribourg, Gambach               7
                       Fribourg, Vuille                7
Name: count, Length: 812, dtype: int64
[7]


In [8]:
o_d_example_15_17 = travel_times_15_17[
    (travel_times_15_17['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_15_17['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_15_17

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,174,2.90,OK
2026-02-12 15:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,176,2.93,OK
2026-02-12 15:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,163,2.72,OK
2026-02-12 16:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 16:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,155,2.58,OK
2026-02-12 16:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,160,2.67,OK
2026-02-12 17:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK


In [9]:
o_d_example_15_17 = travel_times_15_17[
    (travel_times_15_17['origin_name'] == 'Fribourg, Boschung') &
    (travel_times_15_17['dest_name'] == 'Fribourg, Chaley')
]
o_d_example_15_17

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,162,2.70,OK
2026-02-12 15:20:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,160,2.67,OK
2026-02-12 15:40:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,155,2.58,OK
2026-02-12 16:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,152,2.53,OK
2026-02-12 16:20:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,153,2.55,OK
2026-02-12 16:40:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,167,2.78,OK
2026-02-12 17:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,163,2.72,OK


## 1.2 Travel times 17-19:

In [10]:
travel_times_17_19 = pd.read_csv("travel_times_17_19.csv", index_col=0)

In [11]:
origins_17_19 = pd.unique(travel_times_17_19["origin_name"])
print(origins_17_19)

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [12]:
print(travel_times_17_19['origin_name'].value_counts())
print(travel_times_17_19['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  364
Fribourg, Mon-Repos               364
Fribourg, Cité-Jardins            364
Fribourg, Boschung                364
Fribourg, Tilleul/Cathédrale      364
Fribourg, St-Pierre               364
Fribourg/Freiburg, Pl. Gare       364
Villars-sur-Glâne, Méridienne     364
Villars-sur-Glâne, Moncor         364
Villars-sur-Glâne, Belle-Croix    364
Villars-sur-Glâne,Villars-Vert    364
Fribourg, Bertigny                364
Fribourg, Bellevue                364
Fribourg, Schönberg Dunant        364
Fribourg, Guintzet                364
Villars-sur-Glâne,Jean Paul II    364
Villars-sur-Glâne, Hôp. cant.     364
Fribourg, Route-de-Tavel          364
Fribourg, Kessler                 364
Fribourg, Ploetscha               364
Fribourg, Windig                  364
Fribourg, Pont-Zaehringen         364
Fribourg, Charmettes              364
Fribourg, Industrie               364
Fribourg, J. Vogt                 364
Fribourg, Fries                   364


In [13]:
print(travel_times_17_19["dest_name"].value_counts())
print(travel_times_17_19["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               364
Fribourg, Cité-Jardins            364
Fribourg, Boschung                364
Fribourg, Tilleul/Cathédrale      364
Fribourg, St-Pierre               364
Fribourg/Freiburg, Pl. Gare       364
Villars-sur-Glâne, Méridienne     364
Villars-sur-Glâne, Moncor         364
Villars-sur-Glâne, Belle-Croix    364
Fribourg, Chaley                  364
Villars-sur-Glâne,Villars-Vert    364
Fribourg, Bertigny                364
Fribourg, Bellevue                364
Fribourg, Schönberg Dunant        364
Fribourg, Guintzet                364
Villars-sur-Glâne,Jean Paul II    364
Villars-sur-Glâne, Hôp. cant.     364
Fribourg, Route-de-Tavel          364
Fribourg, Kessler                 364
Fribourg, Ploetscha               364
Fribourg, Windig                  364
Fribourg, Pont-Zaehringen         364
Fribourg, Charmettes              364
Fribourg, Industrie               364
Fribourg, J. Vogt                 364
Fribourg, Fries                   364
Fr

In [14]:
print(travel_times_17_19.value_counts(['origin_name', 'dest_name']))
print(travel_times_17_19.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             13
                       Fribourg, Cité-Jardins          13
                       Fribourg, Boschung              13
                       Fribourg, Tilleul/Cathédrale    13
                       Fribourg, St-Pierre             13
                                                       ..
Givisiez, Mont Carmel  Fribourg, Industrie             13
                       Fribourg, J. Vogt               13
                       Fribourg, Fries                 13
                       Fribourg, Gambach               13
                       Fribourg, Vuille                13
Name: count, Length: 812, dtype: int64
[13]


In [15]:
o_d_example_17_19 = travel_times_17_19[
    (travel_times_17_19['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_17_19['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_17_19

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 17:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:10:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:30:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 17:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 17:50:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 18:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,150,2.50,OK
2026-02-12 18:10:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,148,2.47,OK
2026-02-12 18:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,148,2.47,OK


## 1.3 Travel times 19-21:

In [16]:
travel_times_19_21 = pd.read_csv("travel_times_19_21.csv", index_col=0)

In [17]:
origins_19_21 = pd.unique(travel_times_19_21["origin_name"])
print(origins_19_21)

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [18]:
print(travel_times_19_21['origin_name'].value_counts())
print(travel_times_19_21['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  196
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196


In [19]:
print(travel_times_19_21["dest_name"].value_counts())
print(travel_times_19_21["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Fribourg, Chaley                  196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196
Fr

In [20]:
print(travel_times_19_21.value_counts(['origin_name', 'dest_name']))
print(travel_times_19_21.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             7
                       Fribourg, Cité-Jardins          7
                       Fribourg, Boschung              7
                       Fribourg, Tilleul/Cathédrale    7
                       Fribourg, St-Pierre             7
                                                      ..
Givisiez, Mont Carmel  Fribourg, Industrie             7
                       Fribourg, J. Vogt               7
                       Fribourg, Fries                 7
                       Fribourg, Gambach               7
                       Fribourg, Vuille                7
Name: count, Length: 812, dtype: int64
[7]


In [21]:
o_d_example_19_21 = travel_times_19_21[
    (travel_times_19_21['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_19_21['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_19_21

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 19:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 19:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,154,2.57,OK
2026-02-12 19:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,144,2.40,OK
2026-02-12 20:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 20:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 20:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 21:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,158,2.63,OK


The csv files contain departure times between 15-17, 17-19 and 19-21 om respectively. The departures have a frequency of 20 minutes in the 15-17 and 19-21 files, whereas the 17-19 shows 10 minute intervals. For that reason, each OD pairing is present 7 times in the former and 13 times in the latter.

## 1.4 Combined travel times:

In [22]:
travel_times = pd.concat([travel_times_15_17, travel_times_17_19, travel_times_19_21], ignore_index=False)

In [23]:
travel_times.index.name = "departure_time"

In [24]:
travel_times.reset_index(inplace=True)


In [25]:
travel_times.head()

,departure_time,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
0,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589152,"Fribourg, Mon-Repos",46.806711,7.172136,270,35,0.58,42,0.70,OK
1,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589138,"Fribourg, Cité-Jardins",46.809385,7.170446,659,86,1.43,117,1.95,OK
2,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.30,174,2.90,OK
3,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8587255,"Fribourg, Tilleul/Cathédrale",46.806090,7.161261,3788,445,7.42,506,8.43,OK
4,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589161,"Fribourg, St-Pierre",46.803911,7.155266,4335,561,9.35,622,10.37,OK


In [26]:
print(travel_times['origin_name'].value_counts())
print(travel_times['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  756
Fribourg, Mon-Repos               756
Fribourg, Cité-Jardins            756
Fribourg, Boschung                756
Fribourg, Tilleul/Cathédrale      756
Fribourg, St-Pierre               756
Fribourg/Freiburg, Pl. Gare       756
Villars-sur-Glâne, Méridienne     756
Villars-sur-Glâne, Moncor         756
Villars-sur-Glâne, Belle-Croix    756
Villars-sur-Glâne,Villars-Vert    756
Fribourg, Bertigny                756
Fribourg, Bellevue                756
Fribourg, Schönberg Dunant        756
Fribourg, Guintzet                756
Villars-sur-Glâne,Jean Paul II    756
Villars-sur-Glâne, Hôp. cant.     756
Fribourg, Route-de-Tavel          756
Fribourg, Kessler                 756
Fribourg, Ploetscha               756
Fribourg, Windig                  756
Fribourg, Pont-Zaehringen         756
Fribourg, Charmettes              756
Fribourg, Industrie               756
Fribourg, J. Vogt                 756
Fribourg, Fries                   756


In [27]:
travel_times['departure_time'] = pd.to_datetime(travel_times['departure_time'])

In [28]:
travel_times.dtypes["departure_time"]

dtype('<M8[us]')

# 2. Model Preparation

$$\min\sum_{k\in K}\sum_{i\in N}\sum_{j\in N}b_1 c^kt_{ij}x_{ij}^k+\sum_{i\in P}b_2c^0(1-\sum_{k\in K}\sum_{j\in P\cup D}x_{ij}^k)+b_3E[Q(x,w,a,q,\xi)]$$


## 2.1 Setting parameters for model

In [29]:
# Base parameters
mini_bus_number = 6
minibus_capacities = [4, 6, 8]
travel_cost_mini_bus = [40, 30, 20]
num_buses_per_category = [2, 2, 2]
time_interval = 5
c_0 = 100

# Calibration parameters
b_1 = 1.0
b_2 = 1.0
b_3 = 1.0

# Time
time_of_interest = '2026-02-12 15:00:00'

## 2.1 Extracting the node indices

In [30]:
nodes = pd.unique(travel_times["origin_station_id"])
nodes_idx_map = {node: i for i, node in enumerate(nodes)}
nodes_idx = list(nodes_idx_map.values())

In [31]:
nodes_idx_map

{np.int64(8589141): 0,
 np.int64(8589152): 1,
 np.int64(8589138): 2,
 np.int64(8591766): 3,
 np.int64(8587255): 4,
 np.int64(8589161): 5,
 np.int64(8592374): 6,
 np.int64(8589130): 7,
 np.int64(8589131): 8,
 np.int64(8588344): 9,
 np.int64(8577786): 10,
 np.int64(8577785): 11,
 np.int64(8504622): 12,
 np.int64(8589271): 13,
 np.int64(8592375): 14,
 np.int64(8592378): 15,
 np.int64(8592377): 16,
 np.int64(8591767): 17,
 np.int64(8589270): 18,
 np.int64(8589147): 19,
 np.int64(8589158): 20,
 np.int64(8587356): 21,
 np.int64(8577741): 22,
 np.int64(8589154): 23,
 np.int64(8588858): 24,
 np.int64(8589155): 25,
 np.int64(8588351): 26,
 np.int64(8577820): 27,
 np.int64(8587238): 28}

## 2.2 Extracting the travel times

In [32]:
travel_times_filtered = travel_times[travel_times['departure_time'] == time_of_interest]
travel_times_filtered['origin_idx'] = travel_times_filtered['origin_station_id'].map(nodes_idx_map)
travel_times_filtered['dest_idx'] = travel_times_filtered['dest_station_id'].map(nodes_idx_map)


In [33]:
od_travel_time = travel_times_filtered.pivot(index='origin_idx', 
                                  columns='dest_idx', 
                                  values='duration_in_traffic_minutes').fillna(0)

# Show first few rows
od_travel_time.head()

dest_idx,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
origin_idx,,,,,,,,,,,,,,,,,,,,,
0,0.00,0.70,1.95,2.90,8.43,10.37,12.48,14.43,12.57,12.62,...,3.28,4.18,3.00,14.37,15.08,14.93,15.48,10.55,8.92,11.25
1,0.75,0.00,1.22,2.30,7.70,9.60,11.77,13.73,11.88,12.07,...,2.65,3.50,2.37,13.65,14.38,14.23,14.77,9.85,8.22,10.52
2,1.68,0.92,0.00,1.05,6.68,8.52,10.70,12.70,10.87,10.98,...,1.53,2.52,2.70,13.92,14.72,13.33,13.75,8.82,7.15,9.52
3,2.70,1.78,0.85,0.00,5.65,7.50,9.73,11.73,9.87,10.03,...,1.07,1.93,1.58,13.63,14.38,12.33,12.72,7.80,6.13,8.42
4,8.30,7.60,6.68,5.77,0.00,2.13,4.25,13.37,11.25,11.42,...,6.70,7.60,6.62,9.28,10.13,7.95,8.45,4.48,5.32,7.48


In [34]:
# Filter the row for the specific origin and destination
od_pair = travel_times_filtered[
    (travel_times_filtered['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_filtered['dest_name'] == 'Fribourg, Tilleul/Cathédrale')
]

# Include the indices in the output
print(od_pair[['origin_name', 'origin_idx', 'dest_name', 'dest_idx', 'duration_in_traffic_minutes']])

        origin_name  origin_idx                     dest_name  dest_idx  \
3  Fribourg, Chaley           0  Fribourg, Tilleul/Cathédrale         4   

   duration_in_traffic_minutes  
3                         8.43  


In [35]:
# Access the specific OD pair
travel_time = od_travel_time.loc[0, 4]
print(travel_time)

8.43


## 2.3 Minimus indices

In [36]:
bus_idx = list(range(0, mini_bus_number))
bus_idx

[0, 1, 2, 3, 4, 5]

In [37]:
bus_cost = {}
current_bus = 0
for cost, count in zip(travel_cost_mini_bus, num_buses_per_category):
    for _ in range(count):
        bus_cost[current_bus] = cost
        current_bus += 1

bus_capacity = {}
for capacity, count in zip(minibus_capacities, num_buses_per_category):
    for _ in range(count):
        bus_capacity[current_bus] = capacity
        current_bus += 1

In [38]:
bus_cost

{0: 40, 1: 40, 2: 30, 3: 30, 4: 20, 5: 20}

In [39]:
bus_capacity

{6: 4, 7: 4, 8: 6, 9: 6, 10: 8, 11: 8}

## 2.4 Model definition

We'll consider a simple model to construct the MILP formulation and then extend it onto the complete network. The following network will be considered at time t:

<center><img src="Simple_Network.jpeg" width="400" /></center>

In [40]:
N_Base = [[0, 1, 2, 3, 4]]
P = [] # set of pickup nodes
D = [] # set of dropoff nodes
S = [] # set of vehicle initial position nodes
Z = [] # set of vehicle virtual destination nodes
u = [] # Matching relationships
q = [] # Number of passengers in request
q_k = [] # Number of passengers in bus k
M = 1000 # Large number
w = 1 # Wait time + service time at node i
a = [] # arrival time at nodes
w_max = 30 # Maximum waiting time of a vehicle at a node
tep = [] # Earliest pickup time
tlp = [] # Latest pickup time
tld = [] # Latest dropoff time
mu = [] # Node position in tour

### 2.4.1 Set of requests:

A request $r\in R, r=\{1,\ldots,|R|\}$ is represented as a vector: $(q_r, p_r, d_r)$, $q_r$ is the number of passengers in this request, $p_r$ is the pickup node, $d_r$ the drop-off node and $|R|$ is the number of requests considered in the horizon. There are three types of requests:
 - $r\in R_{waiting}$: waiting requests
 - $r\in R_{scheduled}$: scheduled requests (those that were assigned to certain vehicles; passengers could either be waiting or already on the bus)
 - $r\in R_{predicted}$: predicted requests in the future

To build the initial model we'll consider the following requests, which are all in the waiting pool:

In [ ]:
# Requests considered (passengers, origin, destination)
r_1 = [2, 1, 5]
r_2 = [4, 2, 4]
r_3 = [1, 1, 4]
r_4 = [3, 2, 5]
r_5 = [2, 1, 4]
r_6 = [1, 5, 3]
r_7 = [4, 4, 2]
r_8 = [2, 3, 1]
r_9 = [3, 3, 4]
r_10 = [1, 5, 1]
R_wait = [r_1, r_2, r_3, r_4, r_5, r_6, r_7, r_8, r_9, r_10]
R_sched = []
R_pred = []
R = R_wait + R_sched

n_req = len(R)

bus_idx = list(range(0, 2))

In [42]:
print(R)

[[2, 1, 5], [4, 2, 4], [1, 1, 4], [3, 2, 5], [2, 1, 4], [1, 5, 3], [4, 4, 2], [2, 3, 1], [3, 3, 4], [1, 5, 1]]


### 2.4.2 Vehicle attributes:
$K$ is the set of vehicles available during each time step, each vehicle $k\in K, k=\{1,\ldots,|K|\}$ is represented as a vector: $(q_{max}^k,V_0^k,V_d^k)$. In this vector $q_{max}^k$ represents the number of remaining seats, $V_0^k$ the current position, $V_d^k$ the virtual destination and $|K|$ the number of vehicles. The vehicle returns to the virtual destination after an entire day of operation, the distance from all other locations to the virtual destination is set to 0. $c_k$ is the unit travel cost and $c^0$ the penalty for rejecting a request.

In [43]:
# Vehicles (maximum capacity, current position, virtual destination)
k_1 = [8, 3, 0]
k_2 = [6, 5, 0]
K = [k_1, k_2]
bus_idx = list(range(0, len(K)))

bus_cost = {0: 40, 1: 30}
reject_cost = 100

In [44]:
K

[[8, 3, 0], [6, 5, 0]]

### 2.4.3 Node set:
$N=P\cup D\cup S\cup Z$ is defined as the node set and contains all the pickup nodes $P$, dropoff nodes $D$, $S$ vehicle initial positions, $Z$ set of virtual destination nodes.

More specifically, $P$ is composed of of three subsets, the nodes of waiting $P_{wait}$, the nodes of scheduled $P_{sched}$ and predicted passengers $P_{pred}$. $P$ is the union of these subsets $P=P_{wait}\cup P_{sched}\cup P_{pred}$.

In [45]:
#P_wait = [R_wait[1] for R_wait in R_wait]
#P_sched = [R_sched[1] for R_sched in R_sched]
#P_pred = [R_pred[1] for R_pred in R_pred]

#P = P_wait + P_sched
#P

In [46]:
#D_wait = [R_wait[2] for R_wait in R_wait]
#D_sched = [R_sched[2] for R_sched in R_sched]
#D_pred = [R_pred[2] for R_pred in R_pred]

#D = D_wait + D_sched
#D

In [47]:
#P_and_D = P + D
#P_and_D

In [48]:
#S = [K[1] for K in K]
#S

In [49]:
#Z = [K[2] for K in K]
#Z

In [50]:
#N = P_and_D + S + Z
#N

In [51]:
#s = np.array([1, 1, 1, 1, 1])

#t = np.array([[0, 2, 3, 2, 4],
#     [3, 0, 4, 7, 2],
#     [2, 3, 0, 3, 4],
#     [2, 6, 1, 0, 3],
#     [5, 4, 5, 2, 0]])

# Your logical node numbers
#nodes_logical = [1,2,3,4,5]

# Map logical numbers to 0-based matrix indices
#node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
#t_dict = {(i,j): t[node_to_idx[i], node_to_idx[j]] for i in nodes_logical for j in nodes_logical}

#s_dict = {i: s[node_to_idx[i]] for i in nodes_logical}

#w_max = 10

In [52]:
# Creating unique node indices for each requested pickup (also separated into wait and sched)
P_nodes = list(range(n_req))
P_wait = list(range(len(R_wait)))
P_sched = list(range(len(R_wait), len(R_wait) + len(R_sched)))

# Creating unique node indices for each requested dropoff
D_nodes = list(range(n_req, 2*n_req))

# Creating unique node indices for each vehicle initial position and virtual destination
S_nodes = list(range(2*n_req, 2*n_req + len(K)))
Z_nodes = list(range(2*n_req + len(K), 2*n_req + 2*len(K)))

# Creating node sets
P_and_D = P_nodes + D_nodes
N = P_nodes + D_nodes + S_nodes + Z_nodes

In [53]:
# Creating mapping from modeling nodes to physical nodes
P_loc = {i: R[i][1] for i in range(n_req)}
D_loc = {i+n_req: R[i][2] for i in range(n_req)}
S_loc = {S_nodes[k]: K[k][1] for k in range(len(K))}
Z_loc = {Z_nodes[k]: K[k][2] for k in range(len(K))}

node_to_loc = {}
node_to_loc.update(P_loc)
node_to_loc.update(D_loc)
node_to_loc.update(S_loc)
node_to_loc.update(Z_loc)

### 2.4.4 Travel and service times:

$s_i$ denotes the service time of a pickup or dropoff node $i$, $i\in P\cup D$. $t_{ij}$ is the travel time between two notes, $w_{max}$ the maximum waiting time of a vehicle at a station. In this example we'll consider that the service time is constant for all the nodes and is approximately 1 minute, the travel times were defined randomly and the maximum waiting time was considered to be 10 minutes at a node.

In [54]:
s = np.array([0, 1, 1, 1, 1, 1])

t = np.array([[0, 0, 0, 0, 0, 0],
     [0, 0, 2, 3, 2, 4],
     [0, 3, 0, 4, 7, 2],
     [0, 2, 3, 0, 3, 4],
     [0, 2, 6, 1, 0, 3],
     [0, 5, 4, 5, 2, 0]])

node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Create dictionary travel times keyed by logical nodes
t_dict = {(i,j): t[node_to_idx[node_to_loc[i]], node_to_idx[node_to_loc[j]]] for i in N for j in N}

s_dict = {i: s[node_to_idx[node_to_loc[i]]] for i in N}

w_max = 10

For a given node $i\in P$, the time window is $[tep_i, tlp_i]$, where $tep_i$ represents the passenger's earliest pickup time and $[tlp_i]$ the latest pickup time. Similarly, $[ted_i, tld_i]$, where $ted_i$ represent the passenger's earliest dropoff time and $[tld_i]$ the latest dropoff time. $ted_i$ was set to zero if there are no constraints on the earliest dropoff time. $a_{max}\geq 1$ represents the passenger's tolerance for increased travel time compared with the shortest travel time. We'll assume for the sake of modeling that separate arrays following the same order as the requests provides the earliest pick-up times and that the other values are derived using detour tolerance and waiting time at a station.

In [ ]:
req_t_p_wait = [10, 25, 15, 40, 30, 60, 20, 50, 45, 10] 
req_t_p_sched = [] 
req_t_p_pred = [] 

req_t_p = req_t_p_wait + req_t_p_sched

a_max = 2 # Maximum detour tolerated by passenger (2 * travel time)

pax_max_wait = 10 # We assume a 10 minute window for pickup (tlp - tep)

tep = {} 
tlp = {} 
ted = {} 
tld = {} 

for idx, i in enumerate(P_nodes):
    tep[i] = req_t_p[idx]
    tlp[i] = tep[i] + pax_max_wait

for idx, i in enumerate(D_nodes):
    pickup_node = i - n_req 
    travel_time = t_dict[pickup_node, i]
    ted[i] = 0 # We'll assume no constraint on earliest drop-off time in a first step
    tld[i] = tep[pickup_node] + (travel_time * a_max)

e_dict = {}
l_dict = {}

for i in P_nodes:
    e_dict[i] = tep[i]
    l_dict[i] = tlp[i]

for i in D_nodes:
    e_dict[i] = ted[i]
    l_dict[i] = tld[i]

for i in S_nodes + Z_nodes:
    e_dict[i] = 0
    l_dict[i] = 1440 # 1 full day in minutes

### 2.4.5 Capacity

In [55]:
#Q = [R[0] for R in R_wait] + [R[0] for R in R_sched] + [-R[0] for R in R_wait] + [-R[0] for R in R_sched] + [0 for K in K] + [0 for K in K]

#Q

Q = {}

for i in P_nodes:
    Q[i] = R[i][0]

for i in D_nodes:
    Q[i] = -R[i-n_req][0]

for i in S_nodes + Z_nodes:
    Q[i] = 0
    
Q_max = [K[0] for K in K]

ub_dict = {(i, k): Q_max[k] for i in N for k in bus_idx}

### 2.4.6 Matching relationship:
$u_i^k$ represents the matching relationship between vehicle $k$ and node $i$ ensuring that the scheduled request is served by a predetermined vehicle and is defined for $r\in R_{scheduled}$ only. If vehicle $k$ passes through node $i$, then $u_i^k=1$ and $0$ otherwise. To start of this model, we'll assume no matches were made yet.

In [56]:
# Your logical node numbers
nodes_logical = [0,1,2,3,4,5]

u = np.zeros((len(bus_idx), len(nodes_logical)))

# Map logical numbers to 0-based matrix indices
node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Create dictionary travel times keyed by logical nodes
u_dict = {(k, i): u[k, node_to_idx[node_to_loc[i]]] for i in N for k in bus_idx}

In [57]:
u

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

The matrix $A$ is composed of elements $A_i^m$, which take value 1 if the physical station corresponds to node $i$ and 0 otherwise for all $i\in P\cup D$.

In [58]:
M_stations = list(set(node_to_loc.values()))

In [59]:
print(M_stations)

[0, 1, 2, 3, 4, 5]


In [61]:
node_to_loc

{0: 1,
 1: 2,
 2: 1,
 3: 2,
 4: 1,
 5: 5,
 6: 4,
 7: 3,
 8: 3,
 9: 5,
 10: 5,
 11: 4,
 12: 4,
 13: 5,
 14: 4,
 15: 3,
 16: 2,
 17: 1,
 18: 4,
 19: 1,
 20: 3,
 21: 5,
 22: 0,
 23: 0}

In [62]:
A_m = {}

# i is the logical node, m is the physical station
for i in P_and_D:
    for m in M_stations:
        if node_to_loc[i] == m:
            A_m[i, m] = 1
        else:
            A_m[i, m] = 0

In [63]:
A_m

{(0, 0): 0,
 (0, 1): 1,
 (0, 2): 0,
 (0, 3): 0,
 (0, 4): 0,
 (0, 5): 0,
 (1, 0): 0,
 (1, 1): 0,
 (1, 2): 1,
 (1, 3): 0,
 (1, 4): 0,
 (1, 5): 0,
 (2, 0): 0,
 (2, 1): 1,
 (2, 2): 0,
 (2, 3): 0,
 (2, 4): 0,
 (2, 5): 0,
 (3, 0): 0,
 (3, 1): 0,
 (3, 2): 1,
 (3, 3): 0,
 (3, 4): 0,
 (3, 5): 0,
 (4, 0): 0,
 (4, 1): 1,
 (4, 2): 0,
 (4, 3): 0,
 (4, 4): 0,
 (4, 5): 0,
 (5, 0): 0,
 (5, 1): 0,
 (5, 2): 0,
 (5, 3): 0,
 (5, 4): 0,
 (5, 5): 1,
 (6, 0): 0,
 (6, 1): 0,
 (6, 2): 0,
 (6, 3): 0,
 (6, 4): 1,
 (6, 5): 0,
 (7, 0): 0,
 (7, 1): 0,
 (7, 2): 0,
 (7, 3): 1,
 (7, 4): 0,
 (7, 5): 0,
 (8, 0): 0,
 (8, 1): 0,
 (8, 2): 0,
 (8, 3): 1,
 (8, 4): 0,
 (8, 5): 0,
 (9, 0): 0,
 (9, 1): 0,
 (9, 2): 0,
 (9, 3): 0,
 (9, 4): 0,
 (9, 5): 1,
 (10, 0): 0,
 (10, 1): 0,
 (10, 2): 0,
 (10, 3): 0,
 (10, 4): 0,
 (10, 5): 1,
 (11, 0): 0,
 (11, 1): 0,
 (11, 2): 0,
 (11, 3): 0,
 (11, 4): 1,
 (11, 5): 0,
 (12, 0): 0,
 (12, 1): 0,
 (12, 2): 0,
 (12, 3): 0,
 (12, 4): 1,
 (12, 5): 0,
 (13, 0): 0,
 (13, 1): 0,
 (13, 2): 0,
 (13, 3

### 2.4.6 Vehicle capacity

The vehicle capacity is described by multiple variables, one of these is $q_i^k$, which represents the number of passengers in vehicle $k$ when the vehicle arrives at node $i$, $i,j\in N$, $k\in K$.

In [ ]:
stop

NameError: name 'stop' is not defined

### 2.4.6 Model definition:

In [ ]:
# Initializing the model
model_MILP_base = gp.Model("MILP_base")
model_MILP_base.Params.OutputFlag = 1

# Initializing the decision variables
x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")

# Objective funtion to be minimized
obj_expr_trav_cost = gp.quicksum(
    b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
    for i in N
    for j in N
    for k in bus_idx
)

obj_expr_reject_cost = gp.quicksum(
    b_2 * c_0 * (1 - gp.quicksum(x_base[i, j, k] for k in bus_idx for j in P_and_D))
    for i in P_nodes
)

# obj_expr_recourse_cost = b_3 * E

model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

# Constraints

# Previous scheduled request served
for i in P_sched:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
        )

# Each request served at most once
for i in P_wait:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) <= 1
        )

# Ensures same vehicle visits pickup and drop-off nodes of same request
for i in P_nodes:
    d = i + n_req
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[d, j, k] for j in N) == 0
        )

# Flow conservation constraints
for i in P_nodes + D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[j, i, k] for j in N) == 0
        )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
    )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
    )

# Capacity constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
            )

#for i_idx, i in enumerate(N):
#    for k in bus_idx:
#        model_MILP_base.addConstr(
#            q_k[i_idx, k] <= Q_max[k]
#        ) ######## Upper bound constraint is already included in decision variable definition

# Time constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
            )

for k in bus_idx:
    for m in M_stations:
        model_MILP_base.addConstr(
            gp.quicksum(A_m[i, m] * w_k[i, k] for i in P_and_D) <= w_max
        )

for i in P:
    for k in bus_idx:
        model_MILP_base.addConstr(
            e_dict[i] <= a_k[i, k] + w_k[i, k]
        )
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i]
        )

for i in D:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i]
        )

for i in P:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i+n_req, k] - a_k[i, k] <= a_max * t_dict[i, i+n_req]
        )

for i in P:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, i+n_req] <= a_k[i+n_req, k]
        )



Set parameter Username
Set parameter LicenseID to value 2782548
Academic license - for non-commercial use only - expires 2027-02-24
Set parameter OutputFlag to value 1


: 

In [ ]:
STOP

# OLD

In [ ]:
# Requests considered (passengers, origin, destination)
r_1 = [2, 1, 5]
r_2 = [4, 2, 4]
r_3 = [1, 1, 4]
r_4 = [3, 2, 5]
r_5 = [2, 1, 4]
r_6 = [1, 5, 3]
r_7 = [4, 4, 2]
r_8 = [2, 3, 1]
r_9 = [3, 3, 4]
r_10 = [1, 5, 1]
R_wait = [r_1, r_2, r_3, r_4, r_5, r_6, r_7, r_8, r_9, r_10]
R_sched = []
R_pred = []
R = R_wait + R_sched

n_req = len(R)

bus_idx = list(range(0, 2))

# Vehicles (maximum capacity, current position, virtual destination)
k_1 = [8, 3, 0]
k_2 = [6, 5, 0]
K = [k_1, k_2]
bus_idx = list(range(0, len(K)))

bus_cost = {0: 40, 1: 30}
reject_cost = 100

P_nodes = list(range(n_req))
D_nodes = list(range(n_req, 2*n_req))
S_nodes = list(range(2*n_req, 2*n_req + len(K)))
Z_nodes = list(range(2*n_req + len(K), 2*n_req + 2*len(K)))
N = P_nodes + D_nodes + S_nodes + Z_nodes

P_loc = {i: R[i][1] for i in range(n_req)}
D_loc = {i+n_req: R[i][2] for i in range(n_req)}
S_loc = {S_nodes[k]: K[k][1] for k in range(len(K))}
Z_loc = {Z_nodes[k]: K[k][2] for k in range(len(K))}

node_to_loc = {}
node_to_loc.update(P_loc)
node_to_loc.update(D_loc)
node_to_loc.update(S_loc)
node_to_loc.update(Z_loc)

s = np.array([1, 1, 1, 1, 1])

t = np.array([[0, 2, 3, 2, 4],
     [3, 0, 4, 7, 2],
     [2, 3, 0, 3, 4],
     [2, 6, 1, 0, 3],
     [5, 4, 5, 2, 0]])

node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
t_dict = {(i,j): t[node_to_idx[node_to_loc[i]], node_to_idx[node_to_loc[j]]] for i in N for j in N}

s_dict = {i: s[node_to_idx[node_to_loc[i]]] for i in N}

w_max = 10

Q = {}

for i in P_nodes:
    Q[i] = R[i][0]

for i in D_nodes:
    Q[i] = -R[i-n_req][0]

for i in S_nodes + Z_nodes:
    Q[i] = 0
    
Q_max = [K[0] for K in K]

ub_dict = {(i, k): Q_max[k] for i in N for k in bus_idx}

# Your logical node numbers
nodes_logical = [1,2,3,4,5]

u = np.zeros((len(K), len(nodes_logical)))

# Map logical numbers to 0-based matrix indices
node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
u_dict = {(k,i): u[k, node_to_idx[i]] for i in nodes_logical for k in bus_idx}

# Initializing the model
model_MILP_base = gp.Model("MILP_base")
model_MILP_base.Params.OutputFlag = 1

# Initializing the decision variables
x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")

# Objective funtion to be minimized
obj_expr_trav_cost = gp.quicksum(
    b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
    for i in N
    for j in N
    for k in bus_idx
)

obj_expr_reject_cost = gp.quicksum(
    b_2 * c_0 * (1 - gp.quicksum(x_base[i, j, k] for k in bus_idx for j in P_and_D))
    for i in P_nodes
)

# obj_expr_recourse_cost = b_3 * E

model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

# Constraints

# Previous scheduled request served
for i in P_sched:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
        )

# Each request served at most once
for i in P_wait:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) <= 1
        )

# Ensures same vehicle visits pickup and drop-off nodes of same request
for i in P_nodes:
    d = i + n_req
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[d, j, k] for j in N) == 0
        )

# Flow conservation constraints
for i in P_nodes + D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[j, i, k] for j in N) == 0
        )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
    )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
    )

# Capacity constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
            )

#for i_idx, i in enumerate(N):
#    for k in bus_idx:
#        model_MILP_base.addConstr(
#            q_k[i_idx, k] <= Q_max[k]
#        ) ######## Upper bound constraint is already included in decision variable definition

# Time constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
            )
